# Prophet Forecasting on AMI Timeseries

This notebook defines a reusable Prophet forecasting pipeline for a single SKU:
- Load and filter the AMI timeseries data
- Prepare it for Prophet
- Train a Prophet model
- Generate a 12-month forecast
- Perform a simple backtest on the last months
- Visualise results with interactive Plotly figures

In [39]:
import os
import pandas as pd
from prophet import Prophet
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def load_timeseries(base_dir="..", filename="timeseries.csv"):
    data_path = os.path.join(os.path.abspath(base_dir), "data", filename)
    df = pd.read_csv(data_path)
    df["date"] = pd.to_datetime(df["date"])
    return df

def filter_sku(df, sku_id):
    df_sku = df[df["sku"] == sku_id].copy()
    df_sku = df_sku.sort_values("date")
    return df_sku

def to_prophet_format(df_sku):
    prophet_df = df_sku[["date", "qty"]].rename(columns={"date": "ds", "qty": "y"})
    return prophet_df

## Prophet model training and forecasting

The functions below:
- Create and fit a Prophet model
- Generate a future dataframe
- Produce a forecast for a configurable horizon

In [40]:
def train_prophet(prophet_df, yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False):
    m = Prophet(
        yearly_seasonality=yearly_seasonality,
        weekly_seasonality=weekly_seasonality,
        daily_seasonality=daily_seasonality,
    )
    m.fit(prophet_df)
    return m

def make_forecast(model, periods=12, freq="MS"):
    future = model.make_future_dataframe(periods=periods, freq=freq)
    forecast = model.predict(future)
    return future, forecast

## Backtesting

The function below performs a simple backtest:
- Split the SKU time series into train and test using a time cutoff
- Fit Prophet on the train part
- Forecast the test horizon
- Compute MAE and RMSE
- Return both metrics and a joined DataFrame for plotting

In [41]:
def backtest_prophet(df_sku, backtest_months=6):
    df_sku = df_sku.sort_values("date")
    last_date = df_sku["date"].max()
    cutoff = last_date - pd.DateOffset(months=backtest_months)

    train_df = df_sku[df_sku["date"] <= cutoff].copy()
    test_df = df_sku[df_sku["date"] > cutoff].copy()

    train_prophet = to_prophet_format(train_df)
    model = train_prophet_model_for_backtest(train_prophet)
    forecast_df = forecast_for_backtest(model, len(test_df))

    forecast_series = forecast_df.set_index("ds")["yhat"].reindex(test_df["date"])

    eval_df = pd.DataFrame(
        {
            "date": test_df["date"].values,
            "actual": test_df["qty"].values,
            "forecast": forecast_series.values,
        }
    )

    mae = mean_absolute_error(eval_df["actual"], eval_df["forecast"])
    rmse = float(np.sqrt(mean_squared_error(eval_df["actual"], eval_df["forecast"])))

    return {"mae": float(mae), "rmse": rmse, "eval_df": eval_df}

def train_prophet_model_for_backtest(train_prophet):
    m = Prophet()
    m.fit(train_prophet)
    return m

def forecast_for_backtest(model, steps):
    future = model.make_future_dataframe(periods=steps, freq="MS")
    forecast = model.predict(future)
    return forecast

## Plotly visualisations

The functions below:
- Build an interactive forecast figure (history + forecast + intervals)
- Build an interactive backtest figure (actual vs forecast on holdout period)

In [42]:
def make_interactive_forecast_figure(model, forecast, sku_id):
    hist = model.history.copy()
    df = forecast.copy()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=hist["ds"],
            y=hist["y"],
            mode="lines+markers",
            name="History",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=df["ds"],
            y=df["yhat"],
            mode="lines",
            name="Forecast",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=pd.concat([df["ds"], df["ds"][::-1]]),
            y=pd.concat([df["yhat_upper"], df["yhat_lower"][::-1]]),
            fill="toself",
            name="Uncertainty",
            opacity=0.2,
            line=dict(width=0),
            showlegend=True,
        )
    )

    fig.update_layout(
        title=f"Prophet forecast – {sku_id}",
        xaxis_title="Date",
        yaxis_title="Quantity",
        legend_title="Legend",
    )
    return fig

def make_interactive_backtest_figure(eval_df, sku_id):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=eval_df["date"],
            y=eval_df["actual"],
            mode="lines+markers",
            name="Actual",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=eval_df["date"],
            y=eval_df["forecast"],
            mode="lines+markers",
            name="Forecast",
        )
    )
    fig.update_layout(
        title=f"Prophet backtest – {sku_id}",
        xaxis_title="Date",
        yaxis_title="Quantity",
        legend_title="Legend",
    )
    return fig

## End-to-end pipeline

The function below ties everything together:

- Load the timeseries from `../data/timeseries.csv`
- Filter to a selected SKU
- Train a Prophet model on the full history
- Produce a 12-month forecast
- Perform a backtest on the last 6 months
- Return metrics and Plotly figures

In [43]:
def run_full_prophet_pipeline(
    base_dir="..",
    filename="timeseries.csv",
    sku_id="SKU1_001",
    forecast_months=12,
    backtest_months=6,
):
    df = load_timeseries(base_dir=base_dir, filename=filename)
    df_sku = filter_sku(df, sku_id)

    prophet_df = to_prophet_format(df_sku)
    model = train_prophet(prophet_df)

    future, forecast = make_forecast(model, periods=forecast_months, freq="MS")
    forecast_fig = make_interactive_forecast_figure(model, forecast, sku_id)

    backtest_result = backtest_prophet(df_sku, backtest_months=backtest_months)
    backtest_fig = make_interactive_backtest_figure(backtest_result["eval_df"], sku_id)

    result = {
        "model": model,
        "forecast": forecast,
        "forecast_fig": forecast_fig,
        "backtest_metrics": {
            "mae": backtest_result["mae"],
            "rmse": backtest_result["rmse"],
        },
        "backtest_fig": backtest_fig,
        "sku_id": sku_id,
    }
    return result

## Get the results

In [44]:
def run_and_display_for_skus(
    skus,
    base_dir="..",
    filename="timeseries.csv",
    forecast_months=12,
    backtest_months=6,
):
    results = {}
    metrics_rows = []
    artifacts_dir = os.path.join(os.path.abspath(base_dir), "artifacts")
    os.makedirs(artifacts_dir, exist_ok=True)
    for sku_id in skus:
        res = run_full_prophet_pipeline(
            base_dir=base_dir,
            filename=filename,
            sku_id=sku_id,
            forecast_months=forecast_months,
            backtest_months=backtest_months,
        )
        results[sku_id] = res
        m = res["backtest_metrics"]
        metrics_rows.append({"sku_id": sku_id, "mae": m["mae"], "rmse": m["rmse"]})
        print(f"{sku_id} backtest metrics:", m)
        res["forecast_fig"].show()
        res["backtest_fig"].show()
        fc = res["forecast"]
        df_fc = fc[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
        df_fc.to_csv(
            os.path.join(artifacts_dir, f"prophet_forecast_{sku_id}.csv"),
            index=False,
        )
    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(
        os.path.join(artifacts_dir, "prophet_backtest_metrics.csv"),
        index=False,
    )
    return results

skus = ["SKU1_001", "SKU1_002", "SKU1_003", "SKU1_004", "SKU1_005", "SKU1_006"]
prophet_results = run_and_display_for_skus(skus)

21:25:13 - cmdstanpy - INFO - Chain [1] start processing
21:25:13 - cmdstanpy - INFO - Chain [1] done processing
21:25:13 - cmdstanpy - INFO - Chain [1] start processing
21:25:13 - cmdstanpy - INFO - Chain [1] done processing


SKU1_001 backtest metrics: {'mae': 7.090084799877917, 'rmse': 7.650641673286128}


21:25:13 - cmdstanpy - INFO - Chain [1] start processing
21:25:14 - cmdstanpy - INFO - Chain [1] done processing
21:25:14 - cmdstanpy - INFO - Chain [1] start processing
21:25:14 - cmdstanpy - INFO - Chain [1] done processing


SKU1_002 backtest metrics: {'mae': 7.273004358072023, 'rmse': 7.899588737440877}


21:25:14 - cmdstanpy - INFO - Chain [1] start processing
21:25:14 - cmdstanpy - INFO - Chain [1] done processing
21:25:14 - cmdstanpy - INFO - Chain [1] start processing
21:25:14 - cmdstanpy - INFO - Chain [1] done processing


...

SKU1_004 backtest metrics: {'mae': 26.352078711110682, 'rmse': 34.55740078111762}


21:25:14 - cmdstanpy - INFO - Chain [1] start processing
21:25:14 - cmdstanpy - INFO - Chain [1] done processing
21:25:14 - cmdstanpy - INFO - Chain [1] start processing
21:25:14 - cmdstanpy - INFO - Chain [1] done processing


SKU1_005 backtest metrics: {'mae': 9.332098182156656, 'rmse': 12.214793573960804}


21:25:14 - cmdstanpy - INFO - Chain [1] start processing
21:25:14 - cmdstanpy - INFO - Chain [1] done processing
21:25:14 - cmdstanpy - INFO - Chain [1] start processing
21:25:14 - cmdstanpy - INFO - Chain [1] done processing


SKU1_006 backtest metrics: {'mae': 13.006333855684964, 'rmse': 15.606990511195486}
